# NB03c -- n-Alkanes & Isoprenoids: Molecular Fingerprint

Characterizes n-alkane profiles, CPI, isoprenoid ratios, compound depletion
by carbon number, TAR/ACL indices, modal carbon shift, and Peters diagram.

**Depends on:** NB00 (schema), NB01 (measurements, compounds, oils), NB02 (diagnostic_ratios)
**Produces:** `molecular_fingerprint_stats` (CPI, ACL, TAR, modal C), `compound_depletion_summary`, ~8 figures


## PART 0 -- Setup

In [ ]:
import sys, warnings, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from IPython.display import display, Image
warnings.filterwarnings('ignore')

sys.path.insert(0, str(Path('..') / 'notebooks'))
from utils import get_conn, STAGE_COLORS, OILTYPE_COLORS, setup_figure_style, FIG_ROOT

setup_figure_style()
import matplotlib as _mpl
_mpl.rcParams['font.family'] = 'DejaVu Sans'

FIG_DIR = FIG_ROOT / 'nb03c'
# Clear FIG_DIR to remove figures from previous NB03c version
import shutil
if FIG_DIR.exists():
    shutil.rmtree(FIG_DIR)
FIG_DIR.mkdir(parents=True, exist_ok=True)

STAGE_ORDER = ['W0', 'W1', 'W2', 'W3']
STAGE_LABELS = {'W0': 'W0 (fresh)', 'W1': 'W1 (~10%)', 'W2': 'W2 (~20%)', 'W3': 'W3 (~30%)'}
OIL_TYPES_ML = ['crude', 'bitumen_blend', 'refined', 'synthetic']
SEED = 42
STAGE_EVAP = {'W0': 0.00, 'W1': 0.10, 'W2': 0.20, 'W3': 0.30}

PM_CLASS = {
    'n_alkane':  'PM-1 (most labile)',
    'isoprenoid':'PM-3 (labile)',
}

print(f'Setup OK | FIG_DIR: {FIG_DIR}')


In [ ]:
# C02 -- Sanity check + create tables + DELETE existing data (idempotency)
with get_conn() as conn:
    checks = pd.read_sql(
        "SELECT 'oils_eda' AS tbl, COUNT(*) AS n FROM oils WHERE include_in_analysis=1 "
        "UNION ALL SELECT 'compounds_active', COUNT(*) FROM compounds WHERE excluded=0 "
        "UNION ALL SELECT 'measurements', COUNT(*) FROM measurements "
        "UNION ALL SELECT 'n_alkane', COUNT(*) FROM compounds "
        "WHERE compound_group='n_alkane' AND excluded=0 "
        "UNION ALL SELECT 'isoprenoid', COUNT(*) FROM compounds "
        "WHERE compound_group='isoprenoid' AND excluded=0", conn)
    print('=== Sanity check ===')
    print(checks.to_string(index=False))

    conn.execute(
        'CREATE TABLE IF NOT EXISTS molecular_fingerprint_stats ('
        'stat_id INTEGER PRIMARY KEY AUTOINCREMENT, '
        'oil_id INTEGER NOT NULL, '
        'stage_code TEXT NOT NULL, '
        'stat_name TEXT NOT NULL, '
        'stat_value REAL, '
        'FOREIGN KEY (oil_id) REFERENCES oils(oil_id) ON DELETE CASCADE, '
        'UNIQUE(oil_id, stage_code, stat_name))')
    conn.execute(
        'CREATE TABLE IF NOT EXISTS compound_depletion_summary ('
        'compound_name TEXT NOT NULL PRIMARY KEY, '
        'compound_group TEXT NOT NULL, '
        'median_w0 REAL, median_w3 REAL, depletion_pct REAL, '
        'n_oils INTEGER, pm_class TEXT)')

    # DELETE existing NB03c data (idempotency)
    conn.execute('DELETE FROM molecular_fingerprint_stats')
    conn.execute('DELETE FROM compound_depletion_summary')
print('Tables ready (existing data cleared).')


## PART 1 -- n-Alkane Profiles, CPI & Isoprenoid Ratios

In [ ]:
# C03 -- Load n-alkane measurements + extract carbon numbers
with get_conn() as conn:
    df_alk = pd.read_sql(
        'SELECT m.oil_id, o.oil_name, o.oil_type, '
        'm.stage_code AS stage, c.compound_name, '
        'm.value_imputed AS intensity '
        'FROM measurements m '
        'JOIN oils o ON m.oil_id = o.oil_id '
        'JOIN compounds c ON m.compound_id = c.compound_id '
        'WHERE c.compound_group = "n_alkane" AND c.excluded = 0 '
        'AND o.include_in_analysis = 1 '
        'AND m.stage_code IN ("W0","W1","W2","W3")', conn)

df_alk['carbon_n'] = df_alk['compound_name'].str.extract(r'n-C(\d+)').astype(float)
df_alk['stage'] = pd.Categorical(df_alk['stage'], categories=STAGE_ORDER, ordered=True)

print(f'Rows: {len(df_alk):,}')
print(f'Unique n-alkanes: {df_alk["compound_name"].nunique()}')
cn = df_alk['carbon_n'].dropna()
if len(cn) > 0:
    print(f'Carbon range: C{int(cn.min())}--C{int(cn.max())}')
print(f'Oils: {df_alk["oil_id"].nunique()}')
print()
print(df_alk.groupby('oil_type')['oil_id'].nunique().rename('n_oils'))


In [ ]:
# C04 -- F01: Synthetic chromatogram W0 by oil_type
alk_w0 = df_alk[df_alk['stage'] == 'W0'].copy()
tph_sum = alk_w0.groupby('oil_id')['intensity'].transform('sum')
alk_w0['rel'] = alk_w0['intensity'] / tph_sum.replace(0, np.nan)
carbons = sorted(alk_w0['carbon_n'].dropna().unique())

fig, axes = plt.subplots(2, 2, figsize=(14, 8), constrained_layout=True)
for ax, otype in zip(axes.flat, OIL_TYPES_ML):
    sub = alk_w0[alk_w0['oil_type'] == otype]
    medians = [sub[sub['carbon_n'] == c]['rel'].median() for c in carbons]
    ax.bar(range(len(carbons)), medians, color=OILTYPE_COLORS.get(otype, '#aaa'), alpha=0.8)
    ax.set_xticks(range(len(carbons)))
    ax.set_xticklabels([f'C{int(c)}' for c in carbons], rotation=90, fontsize=6)
    ax.set_title(f'{otype} (n={sub["oil_id"].nunique()})', fontsize=10)
    ax.set_ylabel('Relative intensity (median)')
fig.suptitle('n-Alkane Profiles at W0 by Oil Type', fontsize=12)
fig_path = FIG_DIR / 'F01_chromatogram_W0_by_oil_type.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close('all')
display(Image(filename=str(fig_path)))


In [ ]:
# C05 -- F02: Chromatogram evolution W0->W3 (crude median)
alk_crude = df_alk[df_alk['oil_type'] == 'crude'].copy()
tph_sum_c = alk_crude.groupby(['oil_id', 'stage'])['intensity'].transform('sum')
alk_crude['rel'] = alk_crude['intensity'] / tph_sum_c.replace(0, np.nan)
carbons = sorted(alk_crude['carbon_n'].dropna().unique())
x = np.arange(len(carbons))
width = 0.18

fig, ax = plt.subplots(figsize=(14, 5), constrained_layout=True)
for i, stage in enumerate(STAGE_ORDER):
    sub = alk_crude[alk_crude['stage'] == stage]
    medians = [sub[sub['carbon_n'] == c]['rel'].median() for c in carbons]
    ax.bar(x + i * width, medians, width, label=STAGE_LABELS.get(stage, stage),
           color=STAGE_COLORS.get(stage, 'grey'), alpha=0.8)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels([f'C{int(c)}' for c in carbons], rotation=90, fontsize=7)
ax.set_ylabel('Relative intensity (median)')
ax.set_title('n-Alkane Profile Evolution: Crude Oils W0 -> W3')
ax.legend(fontsize=9)
fig_path = FIG_DIR / 'F02_chromatogram_evolution_crude.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close('all')
display(Image(filename=str(fig_path)))
print('Light n-alkanes (C9-C15) deplete with weathering; heavy (C25+) enrich relatively.')


In [ ]:
# C06 -- F03: CPI (Carbon Preference Index) by oil_type and stage
def compute_cpi(df_group, c_min, c_max):
    sub = df_group[(df_group['carbon_n'] >= c_min) & (df_group['carbon_n'] <= c_max)
                   & (df_group['intensity'].notna()) & (df_group['intensity'] > 0)]
    odd  = sub[sub['carbon_n'] % 2 == 1]['intensity'].sum()
    even = sub[sub['carbon_n'] % 2 == 0]['intensity'].sum()
    return odd / even if even > 0 else np.nan

cpi_records = []
for (oil_id, oil_name, oil_type, stage), grp in df_alk.groupby(
        ['oil_id', 'oil_name', 'oil_type', 'stage']):
    cpi_records.append({
        'oil_id': oil_id, 'oil_name': oil_name, 'oil_type': oil_type, 'stage': stage,
        'cpi_short': compute_cpi(grp, 12, 20),
        'cpi_long':  compute_cpi(grp, 21, 35),
        'cpi_total': compute_cpi(grp, 12, 35),
    })
df_cpi = pd.DataFrame(cpi_records)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True, sharey=True)
for ax, (col, title) in zip(axes, [
    ('cpi_short', 'CPI Short (C12-C20)'), ('cpi_long', 'CPI Long (C21-C35)'),
    ('cpi_total', 'CPI Total (C12-C35)')]):
    data = [df_cpi[df_cpi['stage'] == s][col].dropna().values for s in STAGE_ORDER]
    bp = ax.boxplot(data, tick_labels=STAGE_ORDER, patch_artist=True)
    for patch, stage in zip(bp['boxes'], STAGE_ORDER):
        patch.set_facecolor(STAGE_COLORS.get(stage, 'grey'))
    ax.axhline(1.0, color='red', ls='--', alpha=0.7, label='CPI = 1.0')
    ax.set_title(title, fontsize=10); ax.set_xlabel('Stage'); ax.legend(fontsize=8)
axes[0].set_ylabel('CPI')
fig.suptitle('Carbon Preference Index by Weathering Stage', fontsize=12)
fig_path = FIG_DIR / 'F03_CPI_by_oiltype_stage.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close('all')
display(Image(filename=str(fig_path)))
print('CPI_total at W0 by oil_type (median):')
print(df_cpi[df_cpi['stage'] == 'W0'].groupby('oil_type')['cpi_total'].median().round(3).to_string())


# --- KW: does CPI change with weathering stage? ---
from scipy.stats import kruskal
print()
print('=== Kruskal-Wallis: CPI by weathering stage (crude oils) ===')
crude_cpi = df_cpi[df_cpi['oil_type'] == 'crude']
for col, label in [('cpi_short', 'CPI short-chain (C12-C20)'),
                    ('cpi_long',  'CPI long-chain  (C21-C35)'),
                    ('cpi_total', 'CPI total       (C12-C35)')]:
    groups = [crude_cpi[crude_cpi['stage']==s][col].dropna().values for s in STAGE_ORDER]
    groups = [g for g in groups if len(g) >= 3]
    if len(groups) >= 2:
        stat_kw, p_kw = kruskal(*groups)
        sig = '***' if p_kw<0.001 else '**' if p_kw<0.01 else '*' if p_kw<0.05 else 'ns'
        print(f'  {label}: KW p={p_kw:.4f} {sig}')
print('Expected: CPI_short significant, CPI_long not significant')


In [ ]:
# C07 -- F04: nC17/Pr, nC18/Ph, Pr/Ph evolution W0->W3
RATIO_NAMES_ISO = ['nC17_Pr', 'nC18_Ph', 'Pr_Ph']
RATIO_TITLES = {'nC17_Pr': 'nC17/Pr -- process', 'nC18_Ph': 'nC18/Ph -- process',
                'Pr_Ph': 'Pr/Ph -- source (should be stable)'}

with get_conn() as conn:
    ph = ','.join('?' * len(RATIO_NAMES_ISO))
    df_iso = pd.read_sql(
        f'SELECT r.oil_id, o.oil_name, o.oil_type, r.stage_code AS stage, '
        f'r.ratio_name, r.value AS ratio_value '
        f'FROM diagnostic_ratios r JOIN oils o ON r.oil_id = o.oil_id '
        f'WHERE r.ratio_name IN ({ph}) AND o.include_in_analysis = 1 '
        f'AND r.stage_code IN ("W0","W1","W2","W3")', conn, params=RATIO_NAMES_ISO)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
for ax, rname in zip(axes, RATIO_NAMES_ISO):
    sub = df_iso[df_iso['ratio_name'] == rname]
    for otype in OIL_TYPES_ML:
        ot_sub = sub[sub['oil_type'] == otype]
        if ot_sub.empty: continue
        medians = [ot_sub[ot_sub['stage'] == s]['ratio_value'].median() for s in STAGE_ORDER]
        ax.plot(STAGE_ORDER, medians, 'o-', label=otype,
                color=OILTYPE_COLORS.get(otype, 'grey'), markersize=6)
    ax.set_title(RATIO_TITLES.get(rname, rname), fontsize=9)
    ax.set_ylabel('Ratio (median)'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.suptitle('Isoprenoid Diagnostic Ratios: W0 -> W3', fontsize=12)
fig_path = FIG_DIR / 'F04_isoprenoid_ratios_evolution.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close('all')
display(Image(filename=str(fig_path)))


# --- Formal stability test for Pr/Ph ---
from scipy.stats import wilcoxon
print()
print('=== Wilcoxon: Pr/Ph stability W0 vs W3 ===')
with get_conn() as conn:
    df_prph = pd.read_sql("""
        SELECT r.oil_id, r.stage_code AS stage, r.value AS ratio_value
        FROM diagnostic_ratios r
        JOIN oils o ON r.oil_id = o.oil_id
        WHERE r.ratio_name = 'Pr_Ph'
          AND o.include_in_analysis = 1
          AND r.stage_code IN ('W0','W3')
    """, conn)
w0_prph = df_prph[df_prph['stage']=='W0'].set_index('oil_id')['ratio_value']
w3_prph = df_prph[df_prph['stage']=='W3'].set_index('oil_id')['ratio_value']
common_prph = w0_prph.index.intersection(w3_prph.index)
if len(common_prph) >= 5:
    stat_p, p_p = wilcoxon(w0_prph[common_prph].values, w3_prph[common_prph].values)
    if np.isnan(p_p):
        print('  Pr/Ph: ALL DIFFERENCES ZERO -> PERFECTLY STABLE')
    elif p_p > 0.05:
        print(f'  Pr/Ph: p={p_p:.4f} -> STABLE (identity confirmed)')
    else:
        print(f'  Pr/Ph: p={p_p:.4f} -> CHANGES (unexpected)')


In [ ]:
# C08 -- Persist CPI to molecular_fingerprint_stats
with get_conn() as conn:
    records = []
    for _, row in df_cpi.iterrows():
        for stat_name, col in [('cpi_short_chain', 'cpi_short'),
                                ('cpi_long_chain', 'cpi_long'),
                                ('cpi_total', 'cpi_total')]:
            if pd.notna(row[col]):
                records.append((int(row['oil_id']), str(row['stage']), stat_name, float(row[col])))
    conn.executemany(
        'INSERT OR REPLACE INTO molecular_fingerprint_stats '
        '(oil_id, stage_code, stat_name, stat_value) VALUES (?, ?, ?, ?)', records)
print(f'Persisted {len(records):,} CPI stats.')


## PART 2 -- Depletion Analysis (corrected for evaporation mass loss)

In [ ]:
# C_DEP -- Depletion by individual carbon number (corrected for evaporative mass loss)
# ECCC ESTS concentrations are mg/kg of REMAINING oil. As oil loses mass,
# concentrations inflate. Correction: absolute = concentration x remaining_fraction.

def alkane_subgroup(name):
    m = re.match(r'n-C(\d+)', name)
    if not m:
        return 'isoprenoid'
    c = int(m.group(1))
    if c <= 15:  return 'n_alkane_light (C9-C15)'
    if c <= 24:  return 'n_alkane_medium (C16-C24)'
    return 'n_alkane_heavy (C25+)'

PM_SUBCLASS = {
    'n_alkane_light (C9-C15)':  'PM-1 (light, most labile)',
    'n_alkane_medium (C16-C24)':'PM-1 (medium, labile)',
    'n_alkane_heavy (C25+)':    'PM-1 (heavy, moderate)',
    'isoprenoid':               'PM-3 (labile)',
}

with get_conn() as conn:
    df_evap = pd.read_sql("""
        SELECT sp.oil_id, sp.stage_code AS stage, sp.value AS mass_loss_pct
        FROM sample_properties sp JOIN oils o ON sp.oil_id = o.oil_id
        WHERE sp.property_name = 'evaporative_mass_loss'
          AND o.include_in_analysis = 1
          AND sp.stage_code IN ('W0','W1','W2','W3')
    """, conn)

    df_depl_raw = pd.read_sql("""
        SELECT m.oil_id, c.compound_name, c.compound_group,
               m.stage_code AS stage, m.value_imputed AS intensity
        FROM measurements m
        JOIN oils o ON m.oil_id = o.oil_id
        JOIN compounds c ON m.compound_id = c.compound_id
        WHERE c.compound_group IN ('n_alkane', 'isoprenoid')
          AND c.excluded = 0 AND o.include_in_analysis = 1
          AND o.oil_type IN ('crude', 'bitumen_blend')
          AND m.value_imputed IS NOT NULL
          AND m.stage_code IN ('W0', 'W3')
    """, conn)

# Build evap lookup: (oil_id, stage) -> remaining_fraction
df_evap['remaining_frac'] = 1 - df_evap['mass_loss_pct'] / 100.0
evap_lookup = {}
for _, row in df_evap.iterrows():
    evap_lookup[(int(row['oil_id']), row['stage'])] = float(row['remaining_frac'])

# Compute corrected depletion per compound
depl_records = []
for compound_name, grp in df_depl_raw.groupby('compound_name'):
    compound_group = grp['compound_group'].iloc[0]
    subgroup = alkane_subgroup(compound_name)
    w0 = grp[grp['stage'] == 'W0'].groupby('oil_id')['intensity'].mean()
    w3 = grp[grp['stage'] == 'W3'].groupby('oil_id')['intensity'].mean()
    common = w0.index.intersection(w3.index)
    if len(common) < 5:
        continue

    abs_w0_list, abs_w3_list = [], []
    for oid in common:
        c_w0 = float(w0.loc[oid])
        c_w3 = float(w3.loc[oid])
        r_w0 = evap_lookup.get((oid, 'W0'), 1.0)
        r_w3 = evap_lookup.get((oid, 'W3'), 0.70)
        abs_w0_list.append(c_w0 * r_w0)
        abs_w3_list.append(c_w3 * r_w3)

    mean_abs_w0 = np.nanmean(abs_w0_list)
    mean_abs_w3 = np.nanmean(abs_w3_list)
    depl_pct = (mean_abs_w3 - mean_abs_w0) / mean_abs_w0 * 100 if mean_abs_w0 > 0 else np.nan

    m_cn = re.match(r'n-C(\d+)', compound_name)
    carbon_n = int(m_cn.group(1)) if m_cn else None

    depl_records.append({
        'compound_name': compound_name, 'compound_group': compound_group,
        'subgroup': subgroup, 'carbon_n': carbon_n,
        'mean_w0': float(w0.loc[common].mean()), 'mean_w3': float(w3.loc[common].mean()),
        'depletion_pct': depl_pct, 'n_oils': len(common),
        'pm_class': PM_SUBCLASS.get(subgroup, 'unknown'),
    })

depl = pd.DataFrame(depl_records)
print(f'Depletion computed for {len(depl)} compounds.')
print('\nDepletion by subgroup (median %):')
print(depl.groupby('subgroup')['depletion_pct'].agg(['median', 'count']).round(1).to_string())

# Persist
with get_conn() as conn:
    conn.execute('DELETE FROM compound_depletion_summary')
    records = []
    for _, row in depl.iterrows():
        records.append((row['compound_name'], row['compound_group'],
                        float(row['mean_w0']) if pd.notna(row['mean_w0']) else None,
                        float(row['mean_w3']) if pd.notna(row['mean_w3']) else None,
                        float(row['depletion_pct']) if pd.notna(row['depletion_pct']) else None,
                        int(row['n_oils']), row['pm_class']))
    conn.executemany(
        'INSERT INTO compound_depletion_summary '
        '(compound_name, compound_group, median_w0, median_w3, depletion_pct, n_oils, pm_class) '
        'VALUES (?, ?, ?, ?, ?, ?, ?)', records)
print(f'Persisted {len(records)} records.')

# --- Figure ---
depl_alk = depl[depl['carbon_n'].notna()].sort_values('carbon_n')
depl_iso = depl[depl['carbon_n'].isna()]
sg_colors = {'n_alkane_light (C9-C15)': '#d73027',
             'n_alkane_medium (C16-C24)': '#fdae61',
             'n_alkane_heavy (C25+)': '#4575b4'}

fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True)
for sg, color in sg_colors.items():
    sub = depl_alk[depl_alk['subgroup'] == sg]
    ax.scatter(sub['carbon_n'], sub['depletion_pct'], color=color, s=60, label=sg, zorder=3)
ax.plot(depl_alk['carbon_n'], depl_alk['depletion_pct'], 'k-', alpha=0.3, lw=1, zorder=1)

# Isoprenoids as diamonds
for iso_name, x_pos in [('Pristane', 17.3), ('Phytane', 18.3)]:
    val = depl_iso[depl_iso['compound_name'] == iso_name]['depletion_pct'].values
    if len(val) > 0:
        ax.scatter(x_pos, val[0], marker='D', color='#f46d43', s=80, zorder=4, label=iso_name)

ax.axhline(0, color='grey', ls='-', alpha=0.5)
ax.axvline(15.5, color='grey', ls=':', alpha=0.3)
ax.axvline(24.5, color='grey', ls=':', alpha=0.3)
ax.set_xlabel('Carbon number')
ax.set_ylabel('Depletion W0->W3 (%, corrected)')
ax.set_title('n-Alkane Depletion by Carbon Number (corrected for evaporative mass loss)')
ax.legend(fontsize=8, ncol=2)
fig_path = FIG_DIR / 'F_DEP_alkane_depletion_by_carbon.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close('all')
display(Image(filename=str(fig_path)))

# --- Spearman correlation: carbon number vs depletion ---
depl_cn = depl[(depl['compound_group'] == 'n_alkane') & depl['carbon_n'].notna() & depl['depletion_pct'].notna()]
rho_cn, p_cn = stats.spearmanr(depl_cn['carbon_n'], depl_cn['depletion_pct'])
print(f'\nSpearman: carbon number vs depletion')
print(f'  rho = {rho_cn:.3f}, p = {p_cn:.2e}, n = {len(depl_cn)}')
print(f'  Higher carbon number -> less depletion (thermodynamic volatility hierarchy)')

# F_SPEARMAN figure
fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
sg_colors2 = {'n_alkane_light (C9-C15)': '#d73027',
              'n_alkane_medium (C16-C24)': '#f46d43',
              'n_alkane_heavy (C25+)': '#4575b4'}
for sg, grp in depl_cn.groupby('subgroup'):
    ax.scatter(grp['carbon_n'], grp['depletion_pct'],
               color=sg_colors2.get(sg, 'gray'), label=sg, s=70, alpha=0.85,
               edgecolors='k', linewidths=0.5)
z = np.polyfit(depl_cn['carbon_n'], depl_cn['depletion_pct'], 1)
x_range = np.linspace(depl_cn['carbon_n'].min(), depl_cn['carbon_n'].max(), 100)
ax.plot(x_range, np.poly1d(z)(x_range), 'k--', lw=1.5, alpha=0.6)
ax.axhline(0, color='gray', lw=1, ls=':')
ax.set_xlabel('Carbon number')
ax.set_ylabel('Depletion W0->W3 (%, corrected)')
ax.set_title(f'Thermodynamic volatility hierarchy\n'
             f'Spearman rho={rho_cn:.3f}, p={p_cn:.1e}, n={len(depl_cn)}', fontsize=10)
ax.legend(title='Subgroup', fontsize=9)
ax.grid(True, alpha=0.3)
fig_path = FIG_DIR / 'F_SPEARMAN_carbon_depletion.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close('all')
display(Image(filename=str(fig_path)))


# --- Bootstrap 95% CI for Spearman rho ---
np.random.seed(42)
N_BOOT = 1000
boot_rhos = []
for _ in range(N_BOOT):
    idx_b = np.random.choice(len(depl_cn), len(depl_cn), replace=True)
    sample = depl_cn.iloc[idx_b]
    r_b, _ = stats.spearmanr(sample['carbon_n'], sample['depletion_pct'])
    boot_rhos.append(r_b)
ci_low = np.percentile(boot_rhos, 2.5)
ci_high = np.percentile(boot_rhos, 97.5)
print(f'Spearman rho = {rho_cn:.3f}  [95% CI: {ci_low:.3f} - {ci_high:.3f}]')
print(f'Bootstrap N = {N_BOOT}, seed = 42')


### Mass Loss Distribution by Stage


In [ ]:
# C_MASSLOSS -- Distribution of actual evaporative mass loss by stage
with get_conn() as conn:
    df_ml = pd.read_sql("""
        SELECT sp.oil_id, o.oil_name, o.oil_type,
               sp.stage_code AS stage, sp.value AS mass_loss_pct
        FROM sample_properties sp JOIN oils o ON sp.oil_id = o.oil_id
        WHERE sp.property_name = 'evaporative_mass_loss'
          AND o.include_in_analysis = 1
          AND sp.stage_code IN ('W1','W2','W3')
    """, conn)

stages_ml = ['W1', 'W2', 'W3']
nominal = {'W1': 10, 'W2': 20, 'W3': 30}

fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
data_ml = [df_ml[df_ml['stage'] == s]['mass_loss_pct'].dropna().values for s in stages_ml]
vp = ax.violinplot(data_ml, positions=[1, 2, 3], showmedians=True, showextrema=True)
for i, s in enumerate(stages_ml, 1):
    ax.axhline(nominal[s], color='red', ls='--', lw=1, alpha=0.7)
    ax.text(3.3, nominal[s], f'{s} nominal ({nominal[s]}%)', fontsize=8, color='red', va='center')
ax.set_xticks([1, 2, 3])
ax.set_xticklabels(stages_ml)
ax.set_ylabel('Actual mass loss (%)')
ax.set_xlabel('Weathering stage')
ax.set_title('Distribution of Actual Evaporative Mass Loss by Stage\n'
             'ECCC ESTS rotary evaporation 80C (red = nominal values)', fontsize=10)
ax.grid(True, axis='y', alpha=0.3)
fig_path = FIG_DIR / 'F_MASSLOSS_distribution.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close('all')
display(Image(filename=str(fig_path)))

for s in stages_ml:
    sub = df_ml[df_ml['stage'] == s]['mass_loss_pct'].dropna()
    cv = sub.std() / sub.mean() * 100 if sub.mean() > 0 else np.nan
    print(f'{s}: median={sub.median():.1f}%, range=[{sub.min():.1f},{sub.max():.1f}]%, CV={cv:.1f}%')


## PART 3 -- TAR, ACL, Modal Carbon & Peters Diagram

In [ ]:
# C_TAR -- TAR and ACL
# TAR = (nC27+nC29+nC31)/(nC15+nC17+nC19)
# ACL = Sum(n * I_nCn) / Sum(I_nCn) for odd n=25..35

tar_acl_records = []
for (oil_id, oil_name, oil_type, stage), grp in df_alk.groupby(
        ['oil_id', 'oil_name', 'oil_type', 'stage']):
    vals = grp.set_index('carbon_n')['intensity']
    num = sum(vals.get(c, 0) or 0 for c in [27, 29, 31])
    den = sum(vals.get(c, 0) or 0 for c in [15, 17, 19])
    tar = num / den if den > 0 else np.nan
    acl_c = [c for c in range(25, 36) if c % 2 == 1]
    acl_num = sum(c * (vals.get(c, 0) or 0) for c in acl_c)
    acl_den = sum((vals.get(c, 0) or 0) for c in acl_c)
    acl = acl_num / acl_den if acl_den > 0 else np.nan
    tar_acl_records.append({'oil_id': oil_id, 'oil_name': oil_name,
        'oil_type': oil_type, 'stage': stage, 'tar': tar, 'acl': acl})
df_tar_acl = pd.DataFrame(tar_acl_records)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
# TAR by oil_type W0
tar_w0 = df_tar_acl[df_tar_acl['stage'] == 'W0']
data_tar = [tar_w0[tar_w0['oil_type'] == ot]['tar'].dropna().values for ot in OIL_TYPES_ML]
bp = axes[0].boxplot(data_tar, tick_labels=OIL_TYPES_ML, patch_artist=True)
for patch, otype in zip(bp['boxes'], OIL_TYPES_ML):
    patch.set_facecolor(OILTYPE_COLORS.get(otype, '#aaa'))
axes[0].axhline(1.0, color='red', ls='--', alpha=0.7, label='TAR=1')
axes[0].set_title('TAR by Oil Type (W0)'); axes[0].set_ylabel('TAR'); axes[0].legend(fontsize=8)
axes[0].tick_params(axis='x', labelrotation=20)
# ACL evolution
for otype in OIL_TYPES_ML:
    sub = df_tar_acl[df_tar_acl['oil_type'] == otype]
    medians = [sub[sub['stage'] == s]['acl'].median() for s in STAGE_ORDER]
    axes[1].plot(STAGE_ORDER, medians, 'o-', label=otype,
                 color=OILTYPE_COLORS.get(otype, 'grey'), markersize=6)
axes[1].set_title('ACL Evolution W0->W3'); axes[1].set_ylabel('ACL (C25-C35 odd)')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
fig_path = FIG_DIR / 'F_TAR_ACL.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close('all')
display(Image(filename=str(fig_path)))

# Persist
with get_conn() as conn:
    records = []
    for _, row in df_tar_acl.iterrows():
        if pd.notna(row['tar']):
            records.append((int(row['oil_id']), str(row['stage']), 'tar_ratio', float(row['tar'])))
        if pd.notna(row['acl']):
            records.append((int(row['oil_id']), str(row['stage']), 'alkane_centroid', float(row['acl'])))
    conn.executemany(
        'INSERT OR REPLACE INTO molecular_fingerprint_stats '
        '(oil_id, stage_code, stat_name, stat_value) VALUES (?, ?, ?, ?)', records)
print(f'Persisted {len(records)} TAR+ACL records.')
print('TAR at W0 by oil_type (median):')
print(tar_w0.groupby('oil_type')['tar'].median().round(2).to_string())


# --- KW + Wilcoxon: does TAR change with weathering? ---
from scipy.stats import kruskal, wilcoxon
print()
print('=== TAR stability: source or process indicator? ===')
crude_tar_t = df_tar_acl[df_tar_acl['oil_type'] == 'crude']
tar_grps = [crude_tar_t[crude_tar_t['stage']==s]['tar'].dropna().values for s in STAGE_ORDER]
tar_grps = [g for g in tar_grps if len(g) >= 3]
if len(tar_grps) >= 2:
    stat_kw, p_kw = kruskal(*tar_grps)
    sig = '***' if p_kw<0.001 else '**' if p_kw<0.01 else '*' if p_kw<0.05 else 'ns'
    print(f'  KW (crude, all stages): p={p_kw:.4f} {sig}')
tar_w0t = crude_tar_t[crude_tar_t['stage']=='W0'].set_index('oil_id')['tar']
tar_w3t = crude_tar_t[crude_tar_t['stage']=='W3'].set_index('oil_id')['tar']
common_t = tar_w0t.index.intersection(tar_w3t.index)
if len(common_t) >= 5:
    _, p_w = wilcoxon(tar_w0t[common_t].values, tar_w3t[common_t].values)
    direction = 'INCREASES' if tar_w3t[common_t].median() > tar_w0t[common_t].median() else 'DECREASES'
    if np.isnan(p_w):
        print('  Wilcoxon W0 vs W3: STABLE (all differences zero)')
    elif p_w < 0.05:
        print(f'  Wilcoxon W0 vs W3: p={p_w:.4f} -> {direction} (TAR has PROCESS component)')
    else:
        print(f'  Wilcoxon W0 vs W3: p={p_w:.4f} ns -> TAR is SOURCE indicator (stable)')


### UCM x n-Alkane Connection


In [ ]:
# C_UCM -- UCM fraction vs n-alkane depletion (cross-validation NB03b x NB03c)
with get_conn() as conn:
    df_ucm = pd.read_sql("""
        SELECT sp.oil_id, o.oil_name, o.oil_type,
               sp.stage_code AS stage,
               sp.value AS ucm_fraction
        FROM sample_properties sp
        JOIN oils o ON sp.oil_id = o.oil_id
        WHERE sp.property_name = 'ucm_fraction'
          AND o.include_in_analysis = 1
          AND o.oil_type IN ('crude','bitumen_blend','refined','synthetic')
        ORDER BY o.oil_type, o.oil_name, sp.stage_code
    """, conn)
    print(f"UCM rows: {len(df_ucm)}")
    print(df_ucm.head(3))

if df_ucm.empty:
    print('ucm_fraction not found in sample_properties. Skipping UCM analysis.')
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)

    # Panel 1: UCM fraction W0->W3 by oil_type
    ax = axes[0]
    for otype in OIL_TYPES_ML:
        sub = df_ucm[df_ucm['oil_type'] == otype]
        medians = [sub[sub['stage'] == s]['ucm_fraction'].median() for s in STAGE_ORDER]
        ax.plot(STAGE_ORDER, medians, 'o-', label=otype,
                color=OILTYPE_COLORS.get(otype, 'grey'), markersize=6)
    ax.set_title('UCM Fraction W0->W3'); ax.set_ylabel('UCM fraction')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # Panel 2: delta_UCM vs sum depletion of light alkanes per oil
    ax = axes[1]
    ucm_w0 = df_ucm[df_ucm['stage'] == 'W0'].groupby('oil_id')['ucm_fraction'].mean()
    ucm_w3 = df_ucm[df_ucm['stage'] == 'W3'].groupby('oil_id')['ucm_fraction'].mean()
    common_ucm = ucm_w0.index.intersection(ucm_w3.index)
    delta_ucm = ucm_w3.loc[common_ucm] - ucm_w0.loc[common_ucm]
    # Get light alkane depletion per oil from df_depl_raw (already loaded)
    light_names = [f'n-C{c}' for c in range(9, 16)]
    light_raw = df_depl_raw[df_depl_raw['compound_name'].isin(light_names)]
    lw0 = light_raw[light_raw['stage'] == 'W0'].groupby('oil_id')['intensity'].mean()
    lw3 = light_raw[light_raw['stage'] == 'W3'].groupby('oil_id')['intensity'].mean()
    common_l = lw0.index.intersection(lw3.index).intersection(common_ucm)
    if len(common_l) >= 5:
        light_depl = ((lw3.loc[common_l] - lw0.loc[common_l]) / lw0.loc[common_l].replace(0, np.nan) * 100)
        ax.scatter(light_depl, delta_ucm.loc[common_l], s=40, alpha=0.7)
        rho_ucm, p_ucm = stats.spearmanr(light_depl.dropna(), delta_ucm.loc[light_depl.dropna().index])
        ax.set_xlabel('Light alkane depletion C9-C15 (%)')
        ax.set_ylabel('Delta UCM fraction (W3-W0)')
        ax.set_title(f'delta_UCM vs light alkane depletion\nrho={rho_ucm:.2f}', fontsize=9)
    else:
        ax.text(0.5, 0.5, 'Insufficient data', ha='center', va='center', transform=ax.transAxes)

    # Panel 3: UCM_W0 vs CPI_total_W0
    ax = axes[2]
    cpi_w0 = df_cpi[df_cpi['stage'] == 'W0'].set_index('oil_id')['cpi_total']
    common_c = ucm_w0.index.intersection(cpi_w0.index)
    if len(common_c) >= 5:
        ax.scatter(ucm_w0.loc[common_c], cpi_w0.loc[common_c], s=40, alpha=0.7)
        ax.set_xlabel('UCM fraction (W0)'); ax.set_ylabel('CPI total (W0)')
        ax.set_title('UCM W0 vs CPI W0', fontsize=9)
    else:
        ax.text(0.5, 0.5, 'Insufficient data', ha='center', va='center', transform=ax.transAxes)

    fig_path = FIG_DIR / 'F_UCM_connection.png'
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.close('all')
    display(Image(filename=str(fig_path)))
    print(f'UCM fraction at W0 by oil_type (median):')
    print(df_ucm[df_ucm['stage']=='W0'].groupby('oil_type')['ucm_fraction'].median().round(3).to_string())


### n-Alkane Diagnostic Ratios (Top SHAP Features)


In [ ]:
# C_NALK_RATIOS -- Diagnostic n-alkane ratios: evolution W0->W3
# These are the ratios directly used by the XGBoost model (top SHAP features)
NALK_RATIO_NAMES = ['nC10_nC20', 'LMW_HMW_alk', 'nC12_nC24']

with get_conn() as conn:
    ph = ','.join('?' * len(NALK_RATIO_NAMES))
    df_nalk = pd.read_sql(
        f'SELECT r.oil_id, o.oil_type, r.stage_code AS stage, '
        f'r.ratio_name, r.value AS ratio_value '
        f'FROM diagnostic_ratios r JOIN oils o ON r.oil_id = o.oil_id '
        f'WHERE r.ratio_name IN ({ph}) '
        f'AND o.include_in_analysis = 1 '
        f'AND r.stage_code IN ("W0","W1","W2","W3")',
        conn, params=NALK_RATIO_NAMES)

df_nalk['stage'] = pd.Categorical(df_nalk['stage'], STAGE_ORDER, ordered=True)
available_nalk = [r for r in NALK_RATIO_NAMES if r in df_nalk['ratio_name'].unique()]
print(f'N-alkane diagnostic ratios found: {available_nalk}')

RATIO_INFO = {
    'nC10_nC20': ('nC10/nC20', 'process -- 1st significant W0->W1 (top SHAP C1 & C8)'),
    'LMW_HMW_alk': ('LMW/HMW alkanes', 'process -- 1st significant W0->W1'),
    'nC12_nC24': ('nC12/nC24', 'process -- significant W0->W2'),
}

n_r = len(available_nalk)
if n_r == 0:
    print('WARNING: No n-alkane diagnostic ratios found.')
else:
    fig, axes = plt.subplots(1, n_r, figsize=(5 * n_r, 5), constrained_layout=True)
    if n_r == 1:
        axes = [axes]
    for ax, rname in zip(axes, available_nalk):
        label, desc = RATIO_INFO.get(rname, (rname, ''))
        for ot in OIL_TYPES_ML:
            sub = df_nalk[(df_nalk['ratio_name'] == rname) & (df_nalk['oil_type'] == ot)]
            med = [sub[sub['stage'] == s]['ratio_value'].median() for s in STAGE_ORDER]
            ax.plot(STAGE_ORDER, med, 'o-', color=OILTYPE_COLORS.get(ot, 'grey'),
                    label=ot, linewidth=2, markersize=7)
        ax.set_title(f'{label} -- {desc}', fontsize=9)
        ax.set_xlabel('Weathering stage')
        ax.set_ylabel('Ratio value (median)')
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)
        # Print % change
        csub = df_nalk[(df_nalk['ratio_name'] == rname) & (df_nalk['oil_type'] == 'crude')]
        w0m = csub[csub['stage'] == 'W0']['ratio_value'].median()
        w3m = csub[csub['stage'] == 'W3']['ratio_value'].median()
        if pd.notna(w0m) and w0m != 0:
            pct = (w3m - w0m) / w0m * 100
            print(f'{rname}: W0={w0m:.3f} -> W3={w3m:.3f} ({pct:+.1f}%)')
    fig.suptitle('n-Alkane Diagnostic Ratios (Top SHAP Features) -- W0->W3', fontsize=11)
    fig_path = FIG_DIR / 'F_NALK_RATIOS.png'
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.close('all')
    display(Image(filename=str(fig_path)))
    print('All ratios decrease W0->W3 (lighter chains deplete faster).')


In [ ]:
# C_WILCOXON_NALK -- Wilcoxon signed-rank: n-alkane canonical ratios by stage
from scipy.stats import wilcoxon

NALK_RATIOS_TEST = ["nC10_nC20", "LMW_HMW_alk", "nC12_nC24"]
transitions = [("W0","W1"), ("W0","W2"), ("W0","W3")]

with get_conn() as conn:
    df_wlx = pd.read_sql("""
        SELECT r.oil_id, o.oil_type, r.stage_code AS stage,
               r.ratio_name, r.value AS ratio_value
        FROM diagnostic_ratios r
        JOIN oils o ON r.oil_id = o.oil_id
        WHERE r.ratio_name IN ('nC10_nC20', 'LMW_HMW_alk', 'nC12_nC24')
          AND o.include_in_analysis = 1
          AND r.stage_code IN ('W0','W1','W2','W3')
          AND r.value IS NOT NULL
    """, conn)

print(f"Loaded {len(df_wlx)} ratio records")
print("=== Wilcoxon signed-rank: n-alkane canonical ratios ===")
print("Bonferroni correction: alpha/3 = 0.0167 per ratio")
print()

for rname in NALK_RATIOS_TEST:
    sub = df_wlx[df_wlx["ratio_name"] == rname]
    print(f"{rname}:")
    for s1, s2 in transitions:
        v1 = sub[sub["stage"]==s1].groupby("oil_id")["ratio_value"].mean()
        v2 = sub[sub["stage"]==s2].groupby("oil_id")["ratio_value"].mean()
        common = v1.index.intersection(v2.index)
        # Drop pairs where either is NaN
        valid = [(v1.loc[oid], v2.loc[oid]) for oid in common
                 if pd.notna(v1.loc[oid]) and pd.notna(v2.loc[oid])]
        if len(valid) < 5:
            print(f"  {s1}->{s2}: insufficient data (n={len(valid)})")
            continue
        arr1 = np.array([x[0] for x in valid])
        arr2 = np.array([x[1] for x in valid])
        # Check if all differences are zero
        diffs = arr1 - arr2
        if np.all(diffs == 0):
            print(f"  {s1}->{s2}: ALL DIFFERENCES ZERO -> PERFECTLY STABLE (n={len(valid)})")
            continue
        stat, p = wilcoxon(arr1, arr2)
        n = len(valid)
        if np.isnan(p):
            print(f"  {s1}->{s2}: p=nan -> STABLE (n={n})")
        elif p < 0.0167:
            sig = "***" if p<0.001 else "**" if p<0.01 else "*"
            z = abs(stat - n*(n+1)/4) / np.sqrt(n*(n+1)*(2*n+1)/24)
            r_eff = z / np.sqrt(n)
            print(f"  {s1}->{s2}: p={p:.2e} {sig} | r={r_eff:.3f} | n={n}")
        else:
            print(f"  {s1}->{s2}: p={p:.4f} ns | n={n}")


### Wax Content x CPI and TAR


In [ ]:
# C_PR_PH_DEPLETION_TEST -- Mann-Whitney: Pristane vs Phytane depletion
from scipy.stats import mannwhitneyu
print('=== Mann-Whitney U: Pristane vs Phytane depletion ===')
R_W3_test = 1 - STAGE_EVAP['W3']
with get_conn() as conn:
    df_iso_dep = pd.read_sql("""
        SELECT m.oil_id, o.oil_type, m.stage_code AS stage,
               c.compound_name, m.value_imputed AS intensity
        FROM measurements m
        JOIN oils o ON m.oil_id = o.oil_id
        JOIN compounds c ON m.compound_id = c.compound_id
        WHERE c.compound_name IN ('Pristane', 'Phytane')
          AND o.include_in_analysis = 1
          AND o.oil_type IN ('crude', 'bitumen_blend')
          AND m.stage_code IN ('W0', 'W3')
    """, conn)
dep_iso = []
for (oil_id, compound), grp in df_iso_dep.groupby(['oil_id', 'compound_name']):
    w0v = grp[grp['stage'] == 'W0']['intensity'].values
    w3v = grp[grp['stage'] == 'W3']['intensity'].values
    if len(w0v) > 0 and len(w3v) > 0 and w0v[0] > 0:
        dep = (w3v[0] * R_W3_test - w0v[0]) / w0v[0] * 100
        dep_iso.append({'oil_id': oil_id, 'compound': compound, 'depletion': dep})
df_dep_iso = pd.DataFrame(dep_iso)
pr_dep = df_dep_iso[df_dep_iso['compound'] == 'Pristane']['depletion'].dropna()
ph_dep = df_dep_iso[df_dep_iso['compound'] == 'Phytane']['depletion'].dropna()
print(f'Pristane: median={pr_dep.median():.1f}%, n={len(pr_dep)}')
print(f'Phytane:  median={ph_dep.median():.1f}%, n={len(ph_dep)}')
if len(pr_dep) >= 5 and len(ph_dep) >= 5:
    stat_mw, p_mw = mannwhitneyu(pr_dep, ph_dep, alternative='less')
    sig = '***' if p_mw<0.001 else '**' if p_mw<0.01 else '*' if p_mw<0.05 else 'ns'
    print(f'Mann-Whitney U (Pr < Ph): p={p_mw:.4f} {sig}')


In [ ]:
# C_SERIAL_DEPLETION -- Serial correlation: early vs late depletion
print('=== Serial correlation: early vs late depletion rate ===')
R_W1 = 1 - STAGE_EVAP['W1']
R_W3_s = 1 - STAGE_EVAP['W3']
with get_conn() as conn:
    df_serial = pd.read_sql("""
        SELECT m.oil_id, c.compound_name, m.stage_code AS stage,
               m.value_imputed AS intensity
        FROM measurements m
        JOIN oils o ON m.oil_id = o.oil_id
        JOIN compounds c ON m.compound_id = c.compound_id
        WHERE c.compound_group = 'n_alkane' AND c.excluded = 0
          AND o.include_in_analysis = 1
          AND o.oil_type IN ('crude', 'bitumen_blend')
          AND m.stage_code IN ('W0', 'W1', 'W3')
    """, conn)
serial_records = []
for (compound, oil_id), grp in df_serial.groupby(['compound_name', 'oil_id']):
    w0 = grp[grp['stage']=='W0']['intensity'].values
    w1 = grp[grp['stage']=='W1']['intensity'].values
    w3 = grp[grp['stage']=='W3']['intensity'].values
    if not (len(w0) and len(w1) and len(w3)): continue
    w0v, w1v, w3v = w0[0], w1[0], w3[0]
    if not w0v or w0v == 0 or not w1v or w1v == 0: continue
    dep_early = (w1v * R_W1 - w0v) / w0v * 100
    dep_late = (w3v * R_W3_s - w1v * R_W1) / (w1v * R_W1) * 100
    serial_records.append({'compound': compound, 'oil_id': oil_id,
                           'dep_early': dep_early, 'dep_late': dep_late})
df_serial_dep = pd.DataFrame(serial_records).dropna()
if len(df_serial_dep) >= 10:
    rho_s, p_s = stats.spearmanr(df_serial_dep['dep_early'], df_serial_dep['dep_late'])
    print(f'Spearman rho (early vs late): {rho_s:.3f}, p={p_s:.2e}, n={len(df_serial_dep)}')
    if rho_s > 0.5 and p_s < 0.05:
        print('  HIGH correlation -> consistent kinetics')
    elif rho_s < 0.2:
        print('  LOW correlation -> non-linear kinetics')
    else:
        print('  MODERATE correlation -> mixed behavior')


In [ ]:
# C_WAX -- Wax_pct x CPI_long and TAR (cross-validation NB03 x NB03c)
with get_conn() as conn:
    df_wax_c = pd.read_sql("""
        SELECT sp.oil_id, o.oil_name, o.oil_type, sp.value AS wax_pct
        FROM sample_properties sp JOIN oils o ON sp.oil_id = o.oil_id
        WHERE sp.property_name = 'wax_content' AND sp.stage_code = 'W0'
          AND o.include_in_analysis = 1
    """, conn)
    # Average duplicates per oil
    df_wax_c = df_wax_c.groupby(['oil_id', 'oil_name', 'oil_type'])['wax_pct'].mean().reset_index()

# Get CPI_long and TAR from df_cpi and df_tar_acl (already computed)
cpi_long_w0 = df_cpi[df_cpi['stage'] == 'W0'].set_index('oil_id')['cpi_long']
tar_w0_vals = df_tar_acl[df_tar_acl['stage'] == 'W0'].set_index('oil_id')['tar']
wax_vals = df_wax_c.set_index('oil_id')['wax_pct']
otype_map = df_wax_c.set_index('oil_id')['oil_type']

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

# Panel 1: wax_pct vs CPI_long
common1 = wax_vals.index.intersection(cpi_long_w0.dropna().index)
if len(common1) >= 5:
    ax = axes[0]
    for otype in OIL_TYPES_ML:
        ids = [i for i in common1 if otype_map.get(i) == otype]
        if ids:
            ax.scatter(wax_vals.loc[ids], cpi_long_w0.loc[ids],
                       color=OILTYPE_COLORS.get(otype, '#aaa'), label=otype, s=50, alpha=0.7)
    rho1, p1 = stats.spearmanr(wax_vals.loc[common1], cpi_long_w0.loc[common1])
    ax.set_xlabel('Wax content (%)')
    ax.set_ylabel('CPI long-chain (C21-C35)')
    sig1 = '***' if p1 < 0.001 else ('**' if p1 < 0.01 else ('*' if p1 < 0.05 else 'ns'))
    ax.set_title(f'Wax vs CPI_long (rho={rho1:.2f} {sig1})', fontsize=10)
    ax.legend(fontsize=8)
else:
    axes[0].text(0.5, 0.5, 'Insufficient data', ha='center', va='center', transform=axes[0].transAxes)

# Panel 2: wax_pct vs TAR
common2 = wax_vals.index.intersection(tar_w0_vals.dropna().index)
if len(common2) >= 5:
    ax = axes[1]
    for otype in OIL_TYPES_ML:
        ids = [i for i in common2 if otype_map.get(i) == otype]
        if ids:
            ax.scatter(wax_vals.loc[ids], tar_w0_vals.loc[ids],
                       color=OILTYPE_COLORS.get(otype, '#aaa'), label=otype, s=50, alpha=0.7)
    rho2, p2 = stats.spearmanr(wax_vals.loc[common2], tar_w0_vals.loc[common2])
    sig2 = '***' if p2 < 0.001 else ('**' if p2 < 0.01 else ('*' if p2 < 0.05 else 'ns'))
    ax.set_xlabel('Wax content (%)')
    ax.set_ylabel('TAR')
    ax.set_title(f'Wax vs TAR (rho={rho2:.2f} {sig2})', fontsize=10)
    ax.legend(fontsize=8)
else:
    axes[1].text(0.5, 0.5, 'Insufficient data', ha='center', va='center', transform=axes[1].transAxes)

fig_path = FIG_DIR / 'F_WAX_CPI.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close('all')
display(Image(filename=str(fig_path)))

print(f'Wax vs CPI_long: rho={rho1:.2f}, p={p1:.3f}' if len(common1) >= 5 else 'Wax vs CPI_long: insufficient data')
print(f'Wax vs TAR: rho={rho2:.2f}, p={p2:.3f}' if len(common2) >= 5 else 'Wax vs TAR: insufficient data')
print(f'\nOils with highest wax_pct:')
print(df_wax_c.nlargest(5, 'wax_pct')[['oil_name', 'oil_type', 'wax_pct']].to_string(index=False))


In [ ]:
# C_MODAL -- Modal carbon number and shift W0->W3
modal_records = []
for (oil_id, oil_name, oil_type, stage), grp in df_alk.groupby(
        ['oil_id', 'oil_name', 'oil_type', 'stage']):
    valid = grp[(grp['intensity'].notna()) & (grp['intensity'] > 0) & (grp['carbon_n'].notna())]
    if valid.empty: continue
    modal_c = float(valid.loc[valid['intensity'].idxmax(), 'carbon_n'])
    modal_records.append({'oil_id': oil_id, 'oil_name': oil_name,
        'oil_type': oil_type, 'stage': stage, 'modal_carbon': modal_c})
df_modal = pd.DataFrame(modal_records)

fig, axes = plt.subplots(1, len(OIL_TYPES_ML), figsize=(14, 5), sharey=True, constrained_layout=True)
for ax, otype in zip(axes, OIL_TYPES_ML):
    sub = df_modal[df_modal['oil_type'] == otype]
    data = [sub[sub['stage'] == s]['modal_carbon'].dropna().values for s in STAGE_ORDER]
    bp = ax.boxplot(data, tick_labels=STAGE_ORDER, patch_artist=True)
    for patch, stage in zip(bp['boxes'], STAGE_ORDER):
        patch.set_facecolor(STAGE_COLORS.get(stage, 'grey'))
    ax.set_title(otype, fontsize=10)
axes[0].set_ylabel('Modal carbon number')
fig.suptitle('Modal Carbon Shift Under Evaporative Weathering', fontsize=12)
fig_path = FIG_DIR / 'F_MODAL_carbon_shift.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close('all')
display(Image(filename=str(fig_path)))

with get_conn() as conn:
    records = [(int(r['oil_id']), str(r['stage']), 'alkane_modal_carbon', float(r['modal_carbon']))
               for _, r in df_modal.iterrows()]
    conn.executemany(
        'INSERT OR REPLACE INTO molecular_fingerprint_stats '
        '(oil_id, stage_code, stat_name, stat_value) VALUES (?, ?, ?, ?)', records)
print(f'Persisted {len(records)} modal carbon records.')
mc = df_modal[df_modal['oil_type'] == 'crude']
for s in STAGE_ORDER:
    print(f'  crude {s}: modal C{mc[mc["stage"]==s]["modal_carbon"].median():.0f}')


In [ ]:
# C_PETERS -- Peters diagram (nC17/Pr vs Pr/Ph)
with get_conn() as conn:
    df_peters = pd.read_sql(
        'SELECT r.oil_id, o.oil_type, r.stage_code AS stage, '
        'r.ratio_name, r.value AS ratio_value '
        'FROM diagnostic_ratios r JOIN oils o ON r.oil_id = o.oil_id '
        'WHERE r.ratio_name IN ("nC17_Pr", "Pr_Ph") '
        'AND o.include_in_analysis = 1 '
        'AND r.stage_code IN ("W0", "W3")', conn)

pw = df_peters.pivot_table(index=['oil_id', 'oil_type', 'stage'],
    columns='ratio_name', values='ratio_value').reset_index()
pw.columns.name = None

if 'nC17_Pr' not in pw.columns or 'Pr_Ph' not in pw.columns:
    print('nC17_Pr or Pr_Ph not available. Skipping Peters diagram.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
    pw0 = pw[pw['stage'] == 'W0'].dropna(subset=['nC17_Pr', 'Pr_Ph'])
    pw3 = pw[pw['stage'] == 'W3'].dropna(subset=['nC17_Pr', 'Pr_Ph'])

    # Panel 1: W0 by oil_type
    ax = axes[0]
    for otype in OIL_TYPES_ML:
        sub = pw0[pw0['oil_type'] == otype]
        if not sub.empty:
            ax.scatter(sub['nC17_Pr'], sub['Pr_Ph'],
                       color=OILTYPE_COLORS.get(otype, '#aaa'), label=otype, s=60, alpha=0.8)
    ax.axhline(1.0, color='gray', ls='--', alpha=0.5)
    ax.axvline(1.5, color='gray', ls=':', alpha=0.5)
    ax.text(0.95, 0.95, 'Oxic/Terrigenous', fontsize=8, color='gray', ha='right', va='top', transform=ax.transAxes)
    ax.text(0.95, 0.05, 'Marine normal', fontsize=8, color='gray', ha='right', va='bottom', transform=ax.transAxes)
    ax.text(0.05, 0.95, 'Biodegraded', fontsize=8, color='gray', ha='left', va='top', transform=ax.transAxes)
    ax.text(0.05, 0.05, 'Hypersaline/Anoxic', fontsize=8, color='gray', ha='left', va='bottom', transform=ax.transAxes)
    ax.set_xlabel('nC17/Pristane'); ax.set_ylabel('Pr/Ph')
    ax.set_title('Peters Diagram -- W0 (source)'); ax.legend(fontsize=8)

    # Panel 2: W0->W3 arrows
    ax = axes[1]
    for otype in OIL_TYPES_ML:
        s0 = pw0[pw0['oil_type'] == otype].set_index('oil_id')
        s3 = pw3[pw3['oil_type'] == otype].set_index('oil_id')
        common = s0.index.intersection(s3.index)
        color = OILTYPE_COLORS.get(otype, '#aaa')
        for oid in common:
            x0, y0 = float(s0.loc[oid, 'nC17_Pr']), float(s0.loc[oid, 'Pr_Ph'])
            x3, y3 = float(s3.loc[oid, 'nC17_Pr']), float(s3.loc[oid, 'Pr_Ph'])
            ax.annotate('', xy=(x3, y3), xytext=(x0, y0),
                        arrowprops=dict(arrowstyle='->', color=color, lw=0.8, alpha=0.5))
        if len(common) > 0:
            ax.scatter(s0.loc[common, 'nC17_Pr'], s0.loc[common, 'Pr_Ph'],
                       color=color, s=40, alpha=0.6, marker='o')
            ax.scatter(s3.loc[common, 'nC17_Pr'], s3.loc[common, 'Pr_Ph'],
                       color=color, s=40, alpha=0.6, marker='s')
    ax.plot([], [], 'o', color='grey', label='W0')
    ax.plot([], [], 's', color='grey', label='W3')
    ax.axhline(1.0, color='gray', ls='--', alpha=0.5)
    ax.axvline(1.5, color='gray', ls=':', alpha=0.5)
    ax.set_xlabel('nC17/Pristane'); ax.set_ylabel('Pr/Ph')
    ax.set_title('Peters Diagram -- Weathering Trajectories'); ax.legend(fontsize=8)

    fig.suptitle('Peters Cross-Plot: Depositional Environment & Weathering', fontsize=12)
    fig_path = FIG_DIR / 'F_PETERS_diagram.png'
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.close('all')
    display(Image(filename=str(fig_path)))
    print('nC17/Pr decreases W0->W3 (process), Pr/Ph stable (source).')


### Pristane & Phytane: Individual Analysis


In [ ]:
# Pr vs Ph: scatter W0, evolution W0->W3, depletion profile
with get_conn() as conn:
    df_iso_raw = pd.read_sql("""
        SELECT m.oil_id, o.oil_name, o.oil_type,
               m.stage_code AS stage, c.compound_name,
               m.value_imputed AS intensity
        FROM measurements m
        JOIN oils o ON m.oil_id = o.oil_id
        JOIN compounds c ON m.compound_id = c.compound_id
        WHERE c.compound_name IN ('Pristane', 'Phytane')
          AND c.excluded = 0 AND o.include_in_analysis = 1
          AND m.stage_code IN ('W0','W1','W2','W3')
    """, conn)

iso_wide = df_iso_raw.pivot_table(
    index=['oil_id', 'oil_name', 'oil_type', 'stage'],
    columns='compound_name', values='intensity').reset_index()
iso_wide.columns.name = None

fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)

# --- Panel 1: Pr vs Ph scatter at W0, colored by oil_type ---
iw0 = iso_wide[iso_wide['stage'] == 'W0'].dropna(subset=['Pristane', 'Phytane'])
ax = axes[0]
for otype in OIL_TYPES_ML:
    sub = iw0[iw0['oil_type'] == otype]
    if not sub.empty:
        ax.scatter(sub['Pristane'], sub['Phytane'],
                   color=OILTYPE_COLORS.get(otype, '#aaa'), label=otype, s=50, alpha=0.7)
# Diagonal y=x
lim = [0, max(iw0['Pristane'].max(), iw0['Phytane'].max()) * 1.1]
ax.plot(lim, lim, 'k--', alpha=0.4, label='Pr = Ph')
ax.set_xlabel('Pristane intensity')
ax.set_ylabel('Phytane intensity')
ax.set_title('Pristane vs Phytane (W0)', fontsize=10)
ax.legend(fontsize=8)

# --- Panel 2: Pr/Ph ratio evolution W0->W3 per oil_type ---
iso_wide['pr_ph'] = iso_wide['Pristane'] / iso_wide['Phytane'].replace(0, np.nan)
ax = axes[1]
for otype in OIL_TYPES_ML:
    sub = iso_wide[iso_wide['oil_type'] == otype]
    if sub.empty: continue
    medians = [sub[sub['stage'] == s]['pr_ph'].median() for s in STAGE_ORDER]
    ax.plot(STAGE_ORDER, medians, 'o-', label=otype,
            color=OILTYPE_COLORS.get(otype, 'grey'), markersize=6)
# Individual oil trajectories (crude, light grey)
for oil_id, grp in iso_wide[iso_wide['oil_type'] == 'crude'].groupby('oil_id'):
    grp_s = grp.sort_values('stage')
    ax.plot(grp_s['stage'], grp_s['pr_ph'], '-', color='grey', alpha=0.15, lw=0.5)
ax.axhline(1.0, color='red', ls='--', alpha=0.5, label='Pr/Ph = 1')
ax.set_ylabel('Pr/Ph ratio')
ax.set_xlabel('Weathering stage')
ax.set_title('Pr/Ph Ratio Evolution W0->W3\n(individual crude trajectories in grey)', fontsize=10)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# --- Panel 3: Depletion profile Pr vs Ph (corrected) ---
ax = axes[2]
depl_pr = depl[depl['compound_name'] == 'Pristane']['depletion_pct'].values
depl_ph = depl[depl['compound_name'] == 'Phytane']['depletion_pct'].values
compounds_iso = ['Pristane', 'Phytane']
vals_iso = [depl_pr[0] if len(depl_pr) > 0 else np.nan,
            depl_ph[0] if len(depl_ph) > 0 else np.nan]
# Add nC17 and nC18 for context
depl_c17 = depl[depl['compound_name'] == 'n-C17']['depletion_pct'].values
depl_c18 = depl[depl['compound_name'] == 'n-C18']['depletion_pct'].values
compounds_iso = ['n-C17', 'Pristane', 'n-C18', 'Phytane']
vals_iso = [
    depl_c17[0] if len(depl_c17) > 0 else np.nan,
    depl_pr[0] if len(depl_pr) > 0 else np.nan,
    depl_c18[0] if len(depl_c18) > 0 else np.nan,
    depl_ph[0] if len(depl_ph) > 0 else np.nan,
]
colors_iso = ['#d73027', '#f46d43', '#d73027', '#f46d43']
ax.bar(range(len(compounds_iso)), vals_iso, color=colors_iso, alpha=0.8, edgecolor='black')
ax.set_xticks(range(len(compounds_iso)))
ax.set_xticklabels(compounds_iso)
ax.axhline(0, color='grey', ls='-', alpha=0.5)
ax.set_ylabel('Depletion W0->W3 (%, corrected)')
ax.set_title('n-C17 vs Pr, n-C18 vs Ph\n(n-alkanes deplete faster than isoprenoids)', fontsize=10)

fig_path = FIG_DIR / 'F_PR_PH_individual.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close('all')
display(Image(filename=str(fig_path)))

# Print summary
print('Pr/Ph at W0 by oil_type (median):')
print(iw0.groupby('oil_type').apply(
    lambda g: g['Pristane'].median() / g['Phytane'].median()
    if g['Phytane'].median() > 0 else np.nan).round(2).to_string())
print(f'\nDepletion (corrected): nC17={vals_iso[0]:.1f}%, Pr={vals_iso[1]:.1f}%, '
      f'nC18={vals_iso[2]:.1f}%, Ph={vals_iso[3]:.1f}%')
print('Confirms: n-alkanes deplete faster than their isoprenoid counterparts.')


### Completeness Maps: n-Alkane Coverage


In [ ]:
# Completeness maps: oil x compound missingness, and oil x compound x stage

# --- Map 1: Oil x Compound at W0 (binary: detected / not detected) ---
alk_w0_all = df_alk[df_alk['stage'] == 'W0'].copy()
alk_w0_all['detected'] = (alk_w0_all['intensity'].notna()) & (alk_w0_all['intensity'] > 0)

# Pivot: oil_id x compound_name -> detected (1/0)
detect_pivot = alk_w0_all.pivot_table(
    index='oil_id', columns='compound_name', values='detected',
    aggfunc='max', fill_value=False).astype(int)

# Sort compounds by carbon number
cn_order = sorted(detect_pivot.columns, key=lambda x: int(re.search(r'\d+', x).group()) if re.search(r'\d+', x) else 999)
detect_pivot = detect_pivot[cn_order]

# Sort oils by oil_type then by detection count
oil_types_map = df_alk[['oil_id', 'oil_type']].drop_duplicates().set_index('oil_id')['oil_type']
detect_pivot['oil_type'] = detect_pivot.index.map(oil_types_map)
detect_pivot['n_detected'] = detect_pivot[cn_order].sum(axis=1)
detect_pivot = detect_pivot.sort_values(['oil_type', 'n_detected'], ascending=[True, False])
detect_plot = detect_pivot[cn_order]

fig_height = max(8, len(detect_plot) * 0.18)
fig, ax = plt.subplots(figsize=(14, fig_height), constrained_layout=True)
im = ax.imshow(detect_plot.values, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
ax.set_xticks(range(len(cn_order)))
ax.set_xticklabels([c.replace('n-', '') for c in cn_order], rotation=90, fontsize=7)
ax.set_yticks(range(len(detect_plot)))
# Label with oil_type prefix
ylabels = [f"{oil_types_map.get(oid, '?')[:3]}|{oid}" for oid in detect_plot.index]
ax.set_yticklabels(ylabels, fontsize=5)
ax.set_xlabel('n-Alkane compound')
ax.set_ylabel('Oil (sorted by type and completeness)')
ax.set_title('n-Alkane Detection Map at W0 (green=detected, red=missing)', fontsize=11)
plt.colorbar(im, ax=ax, label='Detected', shrink=0.3)
fig_path = FIG_DIR / 'F_COMPLETENESS_oil_compound.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close('all')
display(Image(filename=str(fig_path)))

pct_detected = detect_plot.sum(axis=0) / len(detect_plot) * 100
print('Detection rate by compound (% of oils with nonzero at W0):')
for c, pct in pct_detected.items():
    flag = ' ** LOW' if pct < 50 else ''
    print(f'  {c:12s} {pct:5.1f}%{flag}')

# --- Map 2: Compound x Stage (mean detection rate across oils) ---
alk_all = df_alk.copy()
alk_all['detected'] = (alk_all['intensity'].notna()) & (alk_all['intensity'] > 0)
stage_detect = alk_all.groupby(['compound_name', 'stage'])['detected'].mean().unstack(fill_value=0)
stage_detect = stage_detect.reindex(columns=STAGE_ORDER)
stage_detect = stage_detect.loc[cn_order]

fig, ax = plt.subplots(figsize=(6, max(8, len(cn_order) * 0.3)), constrained_layout=True)
im = ax.imshow(stage_detect.values * 100, aspect='auto', cmap='RdYlGn', vmin=0, vmax=100)
ax.set_xticks(range(len(STAGE_ORDER)))
ax.set_xticklabels(STAGE_ORDER)
ax.set_yticks(range(len(cn_order)))
ax.set_yticklabels([c.replace('n-', '') for c in cn_order], fontsize=8)
ax.set_xlabel('Weathering stage')
ax.set_ylabel('Compound')
ax.set_title('n-Alkane Detection Rate (%) by Compound x Stage', fontsize=11)
plt.colorbar(im, ax=ax, label='% oils with detection', shrink=0.5)

# Annotate values
for yi in range(len(cn_order)):
    for xi in range(len(STAGE_ORDER)):
        val = stage_detect.values[yi, xi] * 100
        ax.text(xi, yi, f'{val:.0f}', ha='center', va='center', fontsize=6,
                color='white' if val < 50 else 'black')

fig_path = FIG_DIR / 'F_COMPLETENESS_compound_stage.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close('all')
display(Image(filename=str(fig_path)))

print('\nLight alkanes (C9-C12) show declining detection W0->W3 = complete evaporation.')
print('Heavy alkanes (C30+) show stable or declining detection = naturally sparse.')


## Summary

In [ ]:
# Summary
import os
with get_conn() as conn:
    n_mfs = conn.execute('SELECT COUNT(*) FROM molecular_fingerprint_stats').fetchone()[0]
    n_cds = conn.execute('SELECT COUNT(*) FROM compound_depletion_summary').fetchone()[0]
    stat_names = [r[0] for r in conn.execute(
        'SELECT DISTINCT stat_name FROM molecular_fingerprint_stats ORDER BY stat_name')]

figs = sorted(f for f in os.listdir(str(FIG_DIR)) if f.endswith('.png'))
print('=' * 60)
print('NB03c -- n-Alkanes & Isoprenoids: Summary')
print('=' * 60)
print(f'molecular_fingerprint_stats: {n_mfs:,} records')
print(f'  stat_names: {stat_names}')
print(f'compound_depletion_summary: {n_cds} records')
print()
print(f'Figures ({len(figs)}):')
for f in figs:
    size = os.path.getsize(str(FIG_DIR / f))
    print(f'  {f:55s} {size/1024:6.1f} KB')
print()
print('Key analyses:')
print('  - F01-F02: n-alkane chromatogram profiles')
print('  - F03: CPI short/long/total by stage')
print('  - F04: Isoprenoid ratios evolution')
print('  - F_DEP + F_SPEARMAN: Depletion by carbon (corrected for mass loss)')
print('  - F_MASSLOSS: Actual mass loss distribution')
print('  - F_TAR_ACL: TAR and ACL indices')
print('  - F_MODAL: Modal carbon shift')
print('  - F_PETERS: Peters diagram')
print('  - F_UCM: UCM connection (cross-validation NB03b)')
print('  - F_WAX_CPI: Wax vs CPI_long and TAR')
print('  - F_PR_PH: Pristane/Phytane individual analysis')
print('  - F_COMPLETENESS: n-Alkane detection maps')
print()
print('NOTE: PAHs -> NB03c2, Biomarkers -> NB03c3, Cross-class -> NB03c4')


In [ ]:
import gc
plt.close('all')
gc.collect()
print('NB03c complete.')
